In [1]:
import numpy as np
import pytz

from Data_query.trino_config import *
from visualisation import *

In [15]:
stop_trino()

Trino service stopping triggered.


In [12]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
sleep(40)

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [4]:
iceberg_sql(f""" select distinct site_id, ac_capacity_kw, state, min_time, max_time from meta_up23c 
            where is_pv=True and flex_export_detected=False and pf_01 > .98 and min_time < timestamp '2024-01-02'
            limit 2""")

,site_id,ac_capacity_kw,state,min_time,max_time
0,261276652,4.2,QLD,2024-01-01,2025-06-22 00:05:00
1,2111538233,5.0,SA,2024-01-01,2025-12-31 23:55:00


In [ ]:
# Some sites are excluded because no CS day profile is detected for them. For example, site 1525233041, cs_day is 2024-01-21. But no ts data is detected for that day.

In [19]:
# big_workers = 1
# workers = 0
# num_workers = max(workers, big_workers)
# ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
# sleep(20)
num_parts = 7


def run_func(args):
    year, month, split_cons, part = args
    # time_filter = f"year = {year} and month = {month}"
    time_filter = f"year = {year}"
    # part_filter = f"postcode % {num_parts} = {part}"
    part_filter = f"site_id % {num_parts} = {part}"
    meta_filters = (
        f"is_pv=True and {split_cons} and flex_export_detected=False and {part_filter}"
    )
    # meta_filters = f"is_pv=True and {split_cons} and flex_export_detected=False and site_id in (699345787)"
    df = iceberg_sql(f"""
                    with data as 
                        (select 
                            site_id, t_stamp,  sum(power*circuit_polarity/S_99)/1000 as P_kw_norm,
                            sum(energy_reactive*circuit_polarity/S_99)/1000*12 as Q_kvar_norm, 
                            avg(voltage) as V
                        from ts join 
                            (select site_id, circuit_id, circuit_polarity, S_99  from meta_up23c where {meta_filters})
                            as m on ts.circuit_id = m.circuit_id
                        where {time_filter} and {part_filter} and is_pv=True and voltage >= 200 and voltage <= 300 and {split_cons}
                        group by site_id, t_stamp
                            ),
                    bom10min as (
                        select 
                            distinct time, b.latitude, b.longitude, surface_global_irradiance as GHI, cloud_type
                        from bom_nci.solar as b
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) as m 
                            on b.latitude = m.n_lat and b.longitude = m.n_long
                        where {time_filter} 
                    ),
                    bom5min as ((select time as time_5min, latitude, longitude, GHI, cloud_type
                    from bom10min) 
                    union all
                    (select date_add('minute', 5, time) as time_5min, latitude, longitude, GHI, cloud_type
                    FROM bom10min ORDER BY time_5min)),
                    daily_cloud AS (
                        SELECT
                            latitude, longitude,
                            date_trunc('day', time + interval '10' hour)   AS day,
                            date_trunc('month', time + interval '10' hour) AS month,
                            sum(cloud_type) AS cloud_sum, 
                            max(GHI) AS max_GHI
                        FROM bom10min
                        GROUP BY
                            1, 2, 3, 4
                    ),
                    clear_sky AS (
                            SELECT day, latitude, longitude
                            FROM (SELECT day, latitude, longitude, cloud_sum, max_GHI, row_number() OVER (
                                                                    PARTITION BY month, latitude, longitude
                                                                    ORDER BY cloud_sum ASC, day ASC
                                                                    ) AS rn
                                FROM daily_cloud 
                            )
                            WHERE rn < 4 and cloud_sum < 60 and max_GHI > 200
                        ),
                    daily_site_days AS (
                            SELECT 
                                n_lat,
                                n_long,
                                date_trunc('day', t_stamp + interval '10' hour) AS day
                            FROM data d join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            on d.site_id = m.site_id
                            group by n_lat, n_long, date_trunc('day', t_stamp + interval '10' hour) 
                        ),
                    nearest_clear_sky_day AS (
                        SELECT
                            dy.n_lat,
                            dy.n_long,
                            dy.day AS actual_day,
                            c.day AS clear_sky_day,
                            row_number() OVER (
                                PARTITION BY dy.n_lat, dy.n_long, dy.day
                                ORDER BY abs(date_diff('day', dy.day, c.day)), date_diff('day', c.day, dy.day)
                            ) AS rn
                        FROM daily_site_days dy JOIN clear_sky c
                            ON dy.n_lat = c.latitude AND dy.n_long = c.longitude
                    ),
                    nearest_clear_sky AS (
                            SELECT
                                n_lat,
                                n_long,
                                actual_day,
                                clear_sky_day
                            FROM nearest_clear_sky_day
                            WHERE rn = 1
                        ),
                    nearest_cs_days AS (
                        SELECT
                            DISTINCT site_id, clear_sky_day AS cs_day 
                        FROM nearest_clear_sky n JOIN (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                            ON n.n_lat = m.n_lat AND n.n_long = m.n_long
                    ),
                    base AS (
                            SELECT
                                d.*,
                                lag(t_stamp) OVER (
                                    PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                    ORDER BY t_stamp
                                ) AS prev_ts
                            FROM data d
                    ),
                    gaps AS (
                        SELECT *,
                            CASE
                                WHEN prev_ts IS NULL THEN 0
                                WHEN t_stamp - prev_ts > interval '30' minute THEN 1
                                ELSE 0
                            END AS gap_start
                        FROM base
                    ),
                    segments AS (
                        SELECT *,
                            sum(gap_start) OVER (
                                PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                ORDER BY t_stamp
                                ROWS UNBOUNDED PRECEDING
                            ) AS segment_id
                        FROM gaps
                    ),
                    nearest_cs_profiles AS (
                        SELECT
                            s.site_id,
                            date_trunc('day', s.t_stamp + interval '10' hour) AS cs_day,
                            (s.t_stamp + interval '10' hour) - date_trunc('day', s.t_stamp + interval '10' hour) AS cs_tod,
                            approx_percentile(P_kw_norm, 0.6) OVER (
                                    PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                        ORDER BY t_stamp
                                        ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                                    ) AS P_kw_norm_cs,
                            approx_percentile(GHI, 0.6) OVER (
                            PARTITION BY s.site_id, date_trunc('day', s.t_stamp + interval '10' hour), segment_id
                                ORDER BY t_stamp
                                ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                            ) AS GHI_cs,
                            cloud_type as cloud_type_cs
                        FROM segments s join nearest_cs_days n on s.site_id = n.site_id and 
                            date_trunc('day', s.t_stamp + interval '10' hour) = n.cs_day
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m on s.site_id = m.site_id
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = s.t_stamp
                    ),
                    strcutured_data AS (
                        SELECT
                            d.site_id,
                            d.t_stamp,
                            date_trunc('day', d.t_stamp + interval '10' hour) AS actual_day,
                            (d.t_stamp + interval '10' hour) - date_trunc('day', d.t_stamp + interval '10' hour) AS actual_tod,
                            d.V,
                            d.Q_kvar_norm,
                            d.P_kw_norm, 
                            sqrt(pow(d.Q_kvar_norm, 2) + pow(d.P_kw_norm, 2)) AS S_norm,
                            GHI,
                            cloud_type,
                            ncs.cs_day,
                            ncs.cs_tod,
                            ncs.P_kw_norm_cs,
                            ncs.GHI_cs, 
                            ncs.cloud_type_cs
                        FROM data d 
                            join (select distinct site_id, n_lat, n_long from meta_up23c where {meta_filters}) m
                                ON d.site_id = m.site_id 
                            join nearest_cs_profiles ncs 
                                on d.site_id = ncs.site_id and ncs.cs_tod = (d.t_stamp + interval '10' hour) - date_trunc('day', d.t_stamp + interval '10' hour)
                            join nearest_clear_sky n on 
                                n.n_lat = m.n_lat AND n.n_long = m.n_long and n.actual_day = date_trunc('day', d.t_stamp + interval '10' hour)
                                and n.clear_sky_day = ncs.cs_day
                            join bom5min b on m.n_lat = b.latitude and m.n_long = b.longitude and b.time_5min = d.t_stamp
                        Where abs(date_diff('day', ncs.cs_day, n.actual_day)) < 45
                            order by d.site_id, d.t_stamp),
                    train_val_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            t_stamp,
                            P_kw_norm,
                            P_kw_norm_cs,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % 30)
                                AS TIME) AS tod_bin,
                            GHI AS x,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y
                        FROM strcutured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05
                            and V <= 253 and (P_kw_norm >= 1 or S_norm < 1.001)
                    ),
                    train_data as (
                        SELECT t.*
                        FROM train_val_data t JOIN split_days s ON t.site_id = s.site_id AND t.actual_day = s.actual_day
                        WHERE s.day_type = 'train'),
                    validation_on_train_data AS (
                        select 
                            t.site_id, 
                            t.t_stamp, 
                            x as GHI, 
                            P_kw_norm, 
                            P_kw_norm_cs * (a + b * x) AS P_kw_norm_est
                        from train_data t 
                            join pv_ghi_model m on t.site_id = m.site_id and t.tod_bin = m.tod_bin
                    )
                    SELECT *
                    FROM validation_on_train_data
                        """)

    #

    # sleep(20)
    # print(f"Completed {time_filter}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}, part {part}, count: {df['site_id'].nunique()}")
    return df


tasks = [
    (year, month, split_cons, part)
    for year in (2024,)
    for month in range(1, 2)
    for split_cons in [f"system.bucket(postcode, 16) = {i}" for i in range(0, 1)]
    for part in range(0, 1)
]
#   for split_cons in ['system.bucket(postcode, 16) > -1'] ]

try:
    res_train = trino_parallel(run_func, tasks, num_workers=1)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
res_train["t_stamp"] = (
    pd.to_datetime(res_train["t_stamp"])
    .dt.tz_localize("utc")
    .dt.tz_convert(pytz.FixedOffset(10 * 60))
)
res_train

Combining results from all tasks.


,site_id,t_stamp,GHI,P_kw_norm,P_kw_norm_est
0,905749026,2024-06-01 09:25:00+10:00,184.53,0.133172,0.257174
1,905749026,2024-06-01 09:20:00+10:00,184.53,0.135893,0.251106
2,905749026,2024-06-01 08:55:00+10:00,141.55,0.104726,0.218657
3,905749026,2024-06-01 08:35:00+10:00,145.79,0.063638,0.202208
4,905749026,2024-06-01 08:20:00+10:00,77.86,0.080136,0.162382
...,...,...,...,...,...
1954773,1974884471,2024-03-27 09:40:00+10:00,580.38,0.520288,0.513397
1954774,1974884471,2024-03-24 17:05:00+10:00,329.90,0.301783,0.302490
1954775,1974884471,2024-03-24 14:40:00+10:00,768.91,0.698172,0.737345
1954776,1974884471,2024-03-23 15:30:00+10:00,642.12,0.617294,0.610404


In [11]:
df0

,site_id,t_stamp,GHI,P_kw_norm,P_kw_norm_est


In [18]:
sample_site_id = 905749026
df0 = res_train.query(f"site_id=={sample_site_id}").reset_index(drop=True)
t0 = df0["t_stamp"].min()
t1 = df0["t_stamp"].max()
# t1 = t0 + pd.Timedelta(days=6)
# start_time = t0.strftime('%Y-%m-%d %H:%M:%S%z')
# end_time = t1.strftime('%Y-%m-%d %H:%M:%S%z')
if t0.time() == pd.Timestamp("00:00:00").time():
    start_time = t0.strftime("%Y-%m-%d %H:%M:%S%z")
else:
    start_time = (
        (t0 + pd.Timedelta(days=1))
        .replace(hour=0, minute=0, second=0)
        .strftime("%Y-%m-%d %H:%M:%S%z")
    )
if t1.time() == pd.Timestamp("00:00:00").time():
    end_time = t1.strftime("%Y-%m-%d %H:%M:%S%z")
else:
    end_time = t1.replace(hour=0, minute=0, second=0).strftime("%Y-%m-%d %H:%M:%S%z")

num_ticks = 24 * 2 + 1
save_as = "Figures/train_mape25.jpeg"
x_label = "time"
y_labels = ["GHI", "Active Power (kW)", "Active Power (kW)"]
plt_config = {
    "GHI": [0, 0, "-", None, None],
    "P_kw_norm": [1, 0, "-", None, None],
    "P_kw_norm_est": [1, 0, "-", None, None],
}

color_nights = False
# color_by = 'group'
color_by = "attribute"
ax_digit = "1.1f"
a = my_plot4(
    start_time,
    end_time,
    df0,
    plt_config=plt_config,
    ax_digit=ax_digit,
    group_attr="site_id",
    time_attr="t_stamp",
    color_nights=color_nights,
    cmap="plasma",
    figsize=[16 / 2.54, 1.3],
    same_scale=1,
    fontsize=5,
    fontname="DejaVu Sans",
    plot_total=False,
    plot_total_func=["sum", [lambda x: max(x), "max"]],
    num_ticks=num_ticks,
    num_yticks=10,
    dpi=200,
    x_format="%H:%M",
    legend_loc=["upper left", "upper right", "center left", "upper left"],
    x_label=x_label,
    y_labels=y_labels,
    color_by=color_by,
    plot_period=np.timedelta64(1, "D"),
    save_as=save_as,
    rotation=60,
    step=0,
    gridwidth=[0.2, 0.2],
    legend_join="-",
    title="",
    legend_i=0,
    title_i=0,
    only1title=0,
    onlyntime=0,
    show=False,
)
a.do()

Maximum supported image dimension is 65500 pixels


OSError: broken data stream when writing image file